# Finetune model on train

model trained after submission deadline

In [1]:
from datasets import Dataset, DatasetDict, load_metric
import numpy as np
from pathlib import Path
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification, set_seed, TrainingArguments, Trainer

In [2]:
set_seed(42)
"""
    learning rate: 1e-5
    batch size: 32
    optimizer: AdamW
    scheduler: linear
    epochs: 78
    max seq: 500
"""

label2id = {
    "O": 0,
    "B-DISORDER": 1,
    "I-DISORDER": 2,
    "B-DRUG": 3,
    "I-DRUG": 4,
    "B-FUNCTION": 5,
    "I-FUNCTION": 6,
}

id2label = {
    0:'O',
    1:'B-DISORDER',
    2:'I-DISORDER', 
    3:'B-DRUG',
    4:'I-DRUG',
    5:'B-FUNCTION',
    6:'I-FUNCTION'
}

In [13]:
JA_PATH = Path('train/ja_train')
JA_DEV = Path('dev/ja_dev')

train_ds = [f.stem for f in JA_PATH.iterdir() if f.suffix == '.txt']
valid_ds = [f.stem for f in JA_DEV.iterdir() if f.suffix == '.txt']

In [6]:
model_checkpoint = "daisaku-s/medtxt_ner_roberta"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
smmmodel = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=7,id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at daisaku-s/medtxt_ner_roberta and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([79, 768]) in the checkpoint and torch.Size([7, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([79]) in the checkpoint and torch.Size([7]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Data

In [14]:
ds = DatasetDict()

ds['train'] = Dataset.from_dict({"id": train_ds})
ds['validation'] = Dataset.from_dict({"id": valid_ds})

In [15]:
corpus = {}
for f in JA_PATH.iterdir():
    if f.suffix == '.txt':
        with f.open() as text:
            fields = text.readlines()[0].split(':')
            keyid = fields[0]
            offset = len(keyid)+1
            data = ':'.join(fields[1:])
            if f.stem in corpus:
                corpus[f.stem]['text'] = data
                corpus[f.stem]['offset'] = len(keyid) + 1
            else:
                corpus[f.stem] = {'text': data,
                                  'offset': len(keyid) + 1,
                                  'ann': []}
    elif f.suffix == '.ann':
        with f.open() as anno:
            for line in anno:
                if line.startswith("T"):
                    fields = line.strip().split("\t")
                    if f.stem in corpus:
                        corpus[f.stem]['ann'].append((fields[1], fields[2]))
                    else:
                        corpus[f.stem] = {'ann': [(fields[1], fields[2])]}

for f in JA_DEV.iterdir():
    if f.suffix == '.txt':
        with f.open() as text:
            fields = text.readlines()[0].split(':')
            keyid = fields[0]
            offset = len(keyid)+1
            data = ':'.join(fields[1:])
            if f.stem in corpus:
                corpus[f.stem]['text'] = data
                corpus[f.stem]['offset'] = len(keyid) + 1
            else:
                corpus[f.stem] = {'text': data,
                                  'offset': len(keyid) + 1,
                                  'ann': []}
    elif f.suffix == '.ann':
        with f.open() as anno:
            for line in anno:
                if line.startswith("T"):
                    fields = line.strip().split("\t")
                    if f.stem in corpus:
                        corpus[f.stem]['ann'].append((fields[1], fields[2]))
                    else:
                        corpus[f.stem] = {'ann': [(fields[1], fields[2])]}

In [16]:
ner2id = {'DISORDER': 1, 'DRUG': 3, 'FUNCTION': 5}
def tokenize_and_align_labels(examples):
    global corpus
    final_labels = []
    texts = [corpus[textid]['text'] for textid in examples["id"]]
    tokenized_inputs = tokenizer(texts, max_length=512)
    for idx, tid in enumerate(examples["id"]):
        new_labels = [0] * len(tokenized_inputs[idx].word_ids)
        for label, token in corpus[tid]['ann']:
            tag, ts, te = label.split(' ')
            labelid = ner2id[tag]
            ts = int(ts) - corpus[tid]['offset']
            te = int(te) - corpus[tid]['offset']
            start = tokenized_inputs[idx].char_to_token(ts)
            end = tokenized_inputs[idx].char_to_token(te - 1)
            new_labels[start: end + 1] = [labelid + 1] * (end - start + 1)  # more ids based on drug, disorder or function
            new_labels[start] = labelid
        for widx, word_id in enumerate(tokenized_inputs[idx].word_ids):
            if word_id is None:
                new_labels[widx] = -100
        final_labels.append(new_labels)
    tokenized_inputs["labels"] = final_labels
    return tokenized_inputs

In [17]:
tokenized_datasets = ds.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Map:   0%|          | 0/168 [00:00<?, ? examples/s]

## Metric

In [18]:
metric = load_metric("seqeval", trust_remote_code=True)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

/tmp/ipykernel_95519/826350436.py:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("seqeval", trust_remote_code=True)


In [19]:
data_collator = DataCollatorForTokenClassification(tokenizer)

## Train

In [20]:
batch_size = 16
args = TrainingArguments(
    "ja_ner",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=50,
    #weight_decay=0.01,
    #fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    push_to_hub=False,
)


trainer = Trainer(
    smmmodel,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/home/vahbuna/miniforge3/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.393707,0.611794,0.536638,0.571757
2,No log,0.276374,0.711637,0.744612,0.727751
3,No log,0.250549,0.730612,0.771552,0.750524
4,No log,0.242320,0.764211,0.782328,0.773163
5,No log,0.244912,0.729516,0.796336,0.761463
6,No log,0.241494,0.750000,0.795259,0.771967
7,No log,0.246852,0.726829,0.802802,0.762929
8,No log,0.255425,0.757390,0.800647,0.778418
9,No log,0.250591,0.757360,0.803879,0.779927
10,No log,0.254330,0.753036,0.801724,0.776618


/home/vahbuna/miniforge3/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/vahbuna/miniforge3/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=1250, training_loss=0.05722007522583008, metrics={'train_runtime': 460.2246, 'train_samples_per_second': 42.588, 'train_steps_per_second': 2.716, 'total_flos': 878899121566224.0, 'train_loss': 0.05722007522583008, 'epoch': 50.0})

In [22]:
trainer.save_model('ja_best')

## Eval on dev set

In [3]:
import torch

In [13]:
out_dir = Path('ft_dev')
for f in JA_DEV.iterdir():
    if f.suffix != '.txt':
        continue
    with f.open() as text:
        annotations = []
        fields = text.readlines()[0].split(':')
        keyid = fields[0]
        offset = len(keyid)+1
        data = ':'.join(fields[1:])
        with torch.inference_mode():
            vecs = tokenizer(data,
                             padding=True, 
                             truncation=True,
                             return_tensors="pt", max_length=512).to('cuda')
            ner_logits = trainer.model(input_ids=vecs["input_ids"], attention_mask=vecs["attention_mask"])
            idx = torch.argmax(ner_logits.logits, dim=2).detach().cpu().numpy().tolist()[0]
            tokens = vecs.tokens()[1: -1]
        labels = [id2label[x] for x in idx][1:-1]
        prev_label = None
        prev_tag = ['', '']
        candidate = []
        start = 0
        for token, label in zip(tokens, labels):
            tag = label.split('-')
            if token.startswith('▁'):
                token = token[1:]
                start += 1
            if tag[0] == 'B':
                candidate= [(start, len(token) + start, token)]
            elif tag[0] != 'B' and len(prev_tag) > 1 and len(tag) > 1 and tag[1] == prev_tag[1]:
                candidate.append((start, len(token) + start, token))
            elif candidate:
                anno = ''.join(ctoken for s, e, ctoken in candidate)
                annotations.append((prev_tag[1], candidate[0][0] + offset - 1, candidate[-1][1] + offset - 1, anno))
                candidate = []
            start += len(token)
            prev_tag = tag
        count = 1
        with (out_dir / (f.stem + '.ann')).open('w') as output:
            for ann in annotations:
                output.write(f"T{count}\t{ann[0]} {ann[1]} {ann[2]}\t{ann[3]}\n")
                count += 1

|ne|tp|fp|fn|precision|recall|f1|
|---|---:|---:|---:|---:|---:|---:|
|DISORDER|328|139|164|0.7024|0.6667|0.6840|
|DRUG|293|31|80|0.9043|0.7855|0.8407|
|FUNCTION|25|28|40|0.4717|0.3846|0.4237|
|all|646|198|284|0.7654|0.6946|0.7283|


|||
|---|---:|
|Task2aMicroP| 0.7654|
|Task2aMicroR| 0.6946|
|Task2aMicroF1| 0.7283|
|Task2aMacroP|0.6928|
|Task2aMacroR| 0.6123|
|Task2aMacroF1| 0.6495|